In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as sf

In [2]:
spark = SparkSession.builder.appName("task1").getOrCreate()
df = spark.read.csv("data/covid-data.csv", header=True)

26/11/16 22:14:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df.printSchema()

root
 |-- iso_code: string (nullable = true)
 |-- continent: string (nullable = true)
 |-- location: string (nullable = true)
 |-- date: string (nullable = true)
 |-- total_cases: string (nullable = true)
 |-- new_cases: string (nullable = true)
 |-- new_cases_smoothed: string (nullable = true)
 |-- total_deaths: string (nullable = true)
 |-- new_deaths: string (nullable = true)
 |-- new_deaths_smoothed: string (nullable = true)
 |-- total_cases_per_million: string (nullable = true)
 |-- new_cases_per_million: string (nullable = true)
 |-- new_cases_smoothed_per_million: string (nullable = true)
 |-- total_deaths_per_million: string (nullable = true)
 |-- new_deaths_per_million: string (nullable = true)
 |-- new_deaths_smoothed_per_million: string (nullable = true)
 |-- reproduction_rate: string (nullable = true)
 |-- icu_patients: string (nullable = true)
 |-- icu_patients_per_million: string (nullable = true)
 |-- hosp_patients: string (nullable = true)
 |-- hosp_patients_per_million

## Задание 1
Выберите 15 стран с наибольшим процентом переболевших на 31 марта (в выходящем датасете необходимы колонки: iso_code, страна, процент переболевших)

In [4]:
(df
 .where(
    df.iso_code == "AFG"
)
 .select(
    df.iso_code,
    df.date,
    df.total_cases,
    df.population,
)
 .show()
)

+--------+----------+-----------+----------+
|iso_code|      date|total_cases|population|
+--------+----------+-----------+----------+
|     AFG|2020-02-24|        1.0|38928341.0|
|     AFG|2020-02-25|        1.0|38928341.0|
|     AFG|2020-02-26|        1.0|38928341.0|
|     AFG|2020-02-27|        1.0|38928341.0|
|     AFG|2020-02-28|        1.0|38928341.0|
|     AFG|2020-02-29|        1.0|38928341.0|
|     AFG|2020-03-01|        1.0|38928341.0|
|     AFG|2020-03-02|        1.0|38928341.0|
|     AFG|2020-03-03|        2.0|38928341.0|
|     AFG|2020-03-04|        4.0|38928341.0|
|     AFG|2020-03-05|        4.0|38928341.0|
|     AFG|2020-03-06|        4.0|38928341.0|
|     AFG|2020-03-07|        4.0|38928341.0|
|     AFG|2020-03-08|        5.0|38928341.0|
|     AFG|2020-03-09|        7.0|38928341.0|
|     AFG|2020-03-10|        8.0|38928341.0|
|     AFG|2020-03-11|       11.0|38928341.0|
|     AFG|2020-03-12|       12.0|38928341.0|
|     AFG|2020-03-13|       13.0|38928341.0|
|     AFG|

In [5]:
task1_result = (df
 .where(
     (sf.month("date") == 3) &
     (sf.dayofmonth("date") == 31)
 )
 .groupBy("iso_code", "location")
 .agg(
     (sf.max(sf.col("total_cases").cast("double")) /
      sf.max(sf.col("population").cast("double")) * 100).alias("per_ill")
 )
 .select("iso_code", "location", "per_ill")
 .orderBy(sf.col("per_ill").desc())
 .limit(15))
task1_result.show()

+--------+-------------+------------------+
|iso_code|     location|           per_ill|
+--------+-------------+------------------+
|     AND|      Andorra|15.543907331909661|
|     MNE|   Montenegro|14.523725364693293|
|     CZE|      Czechia|14.308848404077997|
|     SMR|   San Marino|13.937179562732041|
|     SVN|     Slovenia|10.370805779121204|
|     LUX|   Luxembourg| 9.847342390123583|
|     ISR|       Israel| 9.625106044786802|
|     USA|United States| 9.203010995860707|
|     SRB|       Serbia| 8.826328557933492|
|     BHR|      Bahrain| 8.488860079114566|
|     PAN|       Panama| 8.228739065460761|
|     PRT|     Portugal| 8.058699735120369|
|     EST|      Estonia| 8.022681579659551|
|     SWE|       Sweden| 7.969744347858805|
|     LTU|    Lithuania| 7.938864728274825|
+--------+-------------+------------------+



In [6]:
(task1_result
 .write
 .format("csv")
 .option("header", "true")
 .mode("overwrite")
 .save("results/task1_result.csv"))

## Задание 2
Top 10 стран с максимальным зафиксированным кол-вом новых случаев за последнюю неделю марта 2021 в отсортированном порядке по убыванию
(в выходящем датасете необходимы колонки: число, страна, кол-во новых случаев)

In [7]:
from pyspark.sql.window import Window

In [8]:
w = Window.partitionBy("location").orderBy(sf.col("new_cases").cast("double").desc())

task2_result = (df
.where(
    (sf.col("date") >= "2021-03-25") &
    (sf.col("date") <= "2021-03-31")
).withColumn("rn", sf.row_number().over(w))
.where(
    sf.col("rn") == 1
).select(
    "date",
    "location",
    "new_cases",
).orderBy(
    sf.col("new_cases").cast("double").desc()
).limit(10)
)
task2_result.show()

+----------+--------------+---------+
|      date|      location|new_cases|
+----------+--------------+---------+
|2021-03-31|         World| 683205.0|
|2021-03-31|        Europe| 255985.0|
|2021-03-31|European Union| 216452.0|
|2021-03-31|          Asia| 183350.0|
|2021-03-25| South America| 148476.0|
|2021-03-25|        Brazil| 100158.0|
|2021-03-26| North America|  93350.0|
|2021-03-26| United States|  77321.0|
|2021-03-31|         India|  72330.0|
|2021-03-31|        France|  59054.0|
+----------+--------------+---------+



In [9]:
(task2_result
 .write
 .format("csv")
 .option("header", "true")
 .mode("overwrite")
 .save("results/task2_result.csv"))

## Задание 3
Посчитайте изменение случаев относительно предыдущего дня в России за последнюю неделю марта 2021. (например: в россии вчера было 9150 , сегодня 8763, итог: -387) (в выходящем датасете необходимы колонки: число, кол-во новых случаев вчера, кол-во новых случаев сегодня, дельта)

In [10]:
w = Window.orderBy(sf.col("date"))

task3_result = (df
.where(
    (sf.col("date") >= "2021-03-25") &
    (sf.col("date") <= "2021-03-31") &
    (sf.col("location") == "Russia")
).withColumn(
    "previous_new_cases",
    sf.lag("new_cases").over(w)
).select(
    "date",
    "previous_new_cases",
    "new_cases",
    (sf.col("new_cases").cast("double") - sf.col("previous_new_cases").cast("double")).alias("delta")
)
)
task3_result.show()

26/11/16 22:14:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/16 22:14:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/16 22:14:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/16 22:14:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/16 22:14:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [11]:
(task3_result
 .write
 .format("csv")
 .option("header", "true")
 .mode("overwrite")
 .save("results/task3_result.csv"))

26/11/16 22:14:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/16 22:14:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
